# AETHER — Training Progress and Model Results

Reads the artifacts a training run writes and plots them. Safe to re-run **while
training is still going**: `metrics.jsonl` is appended and flushed after every
evaluation, so each re-run shows the curves so far.

Point `RUN_DIR` at the run you want. It expects:

```
<RUN_DIR>/metrics.jsonl          one row per evaluation
<RUN_DIR>/config.json            resolved run configuration
<RUN_DIR>/test_metrics.json      final test metrics      (written at the end)
<RUN_DIR>/test_predictions.csv   per-row predictions     (written at the end)
```

The test sections skip themselves if the run has not finished yet. Each cell
produces one standalone 600 dpi PNG in this folder.

In [1]:
from pathlib import Path
import json
import logging
import os
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch

# matplotlib reports a missing font through the logging module rather than
# warnings, so filterwarnings alone never silences "findfont: Font family
# 'Arial' not found".
warnings.filterwarnings("ignore", message=".*findfont.*")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

# ---------------------------------------------------------------------
# plot style
# ---------------------------------------------------------------------
# Arial is the house font, but it ships with no Linux distribution and is not
# installed on this machine. Declaring a fallback chain means the figures use
# Arial wherever it exists and a metrically similar sans elsewhere, rather than
# matplotlib silently substituting whatever it has. Liberation Sans and Nimbus
# Sans share Arial's metrics, so text occupies the same space; DejaVu Sans is
# matplotlib's bundled last resort and is slightly wider.
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = [
    "Arial", "Liberation Sans", "Nimbus Sans", "Helvetica", "DejaVu Sans",
]
mpl.rcParams["font.size"] = 10
DPI = 600
FIGSIZE = (6, 3.8)   # every figure stands alone; no subplots

BAR_COLOR = "#4b5563"
LIGHT_BAR_COLOR = "#d1d5db"
REF_LINE = "#6b7280"
GRID_COLOR = "#f0f0f0"

# ink scale for text that is not an axis label
INK = "#111827"
INK_SOFT = "#4b5563"
INK_MUTED = "#6b7280"

TAB = {"blue": "#4e79a7", "orange": "#f28e2b", "red": "#e15759", "teal": "#76b7b2",
       "green": "#59a14f", "yellow": "#edc948", "purple": "#b07aa1",
       "pink": "#ff9da7", "brown": "#9c755f", "gray": "#bab0ac"}

# roles, so a plot is written against what the color means
BINDER, NON_BINDER = TAB["blue"], TAB["orange"]
TRAIN_COLOR, VAL_COLOR = TAB["blue"], TAB["red"]
CLASS_COLORS = {"Binder": BINDER, "Non-binder": NON_BINDER}


def new_fig(figsize=FIGSIZE):
    """one standalone figure with its axes styled"""
    fig, ax = plt.subplots(figsize=figsize)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(color=GRID_COLOR, linewidth=0.6)
    ax.set_axisbelow(True)
    return fig, ax


def save_fig(fig, name):
    """600 dpi png into figures/, cropped to everything drawn"""
    path = FIG_DIR / name
    fig.savefig(path, dpi=DPI, bbox_inches="tight")
    print(f"saved {path}")


def left_legend(ax, handles, title, anchor):
    """left-aligned legend outside the axes; call again to stack another one below"""
    if ax.get_legend() is not None:
        previous = ax.get_legend()
        ax.add_artist(previous)
        previous.set_clip_on(False)
    legend = ax.legend(handles=handles, title=title, frameon=False,
                       bbox_to_anchor=anchor, loc="upper left")
    legend._legend_box.align = "left"
    legend.get_title().set_ha("left")
    return legend


def class_handles():
    return [Patch(facecolor=c, label=k) for k, c in CLASS_COLORS.items()]


# ---------------------------------------------------------------------
# paths, relative to this notebook
# ---------------------------------------------------------------------
FIG_DIR = Path.cwd()
REPO = FIG_DIR
while REPO != REPO.parent and not (REPO / "data").is_dir():
    REPO = REPO.parent
print("repo:", REPO)

repo: /home/vivian.chu/AETHER


In [2]:
# ---------------------------------------------------------------------
# point this at your run
# ---------------------------------------------------------------------
# train.py --render_figures sets AETHER_RUN_DIR when it executes this notebook
# itself; running the notebook by hand falls back to the default below.
RUN_DIR = Path(os.environ.get("AETHER_RUN_DIR") or (REPO / "outputs" / "classI_esm2_650m"))

if not RUN_DIR.exists():
    found = sorted((REPO / "outputs").glob("*/metrics.jsonl")) if (REPO / "outputs").exists() else []
    raise SystemExit(
        f"No run at {RUN_DIR}.\n" +
        ("Available runs:\n  " + "\n  ".join(str(p.parent) for p in found)
         if found else "No runs under outputs/ — train a model first.")
    )

cfg = json.loads((RUN_DIR / "config.json").read_text())
hist = pd.read_json(RUN_DIR / "metrics.jsonl", lines=True)

val     = hist[hist["phase"] == "val"]          # full end-of-epoch validation
partial = hist[hist["phase"] == "val_partial"]  # mid-epoch, capped batches

# Epochs here are ~5,400 steps, so early in a run there are only mid-epoch
# checks. Fall back to those as the main series so the notebook is useful while
# the first epoch is still going; they are noisier (capped batch count).
CURVE_IS_PARTIAL = len(val) == 0
curve = partial if CURVE_IS_PARTIAL else val

print(f"run      : {RUN_DIR.name}")
print(f"backbone : {cfg['model_name']}  (frozen={cfg['freeze_backbone']})")
print(f"batch    : {cfg['batch_size']}   lr={cfg['lr']}   epochs={cfg['epochs']}")
print(f"logged   : {len(val)} full validations, {len(partial)} mid-epoch checks")
if CURVE_IS_PARTIAL:
    print("note     : no full epoch finished yet — plotting mid-epoch checks")
hist.tail(5)

run      : tcr_phase2
backbone : facebook/esm2_t33_650M_UR50D  (frozen=True)
batch    : 64   lr=0.0003   epochs=20
logged   : 7 full validations, 14 mid-epoch checks


,phase,epoch,step,global_step,train_loss,lr,val_loss,val_mae,val_mse,val_auc,val_auprc,val_acc
16,val_partial,6,1000,7380,0.113425,0.000244,0.651187,0.185987,0.157164,0.614705,0.237858,0.832812
17,val,6,1276,7656,0.114065,0.000240,0.625431,0.180757,0.154322,0.756433,0.331013,0.835345
18,val_partial,7,500,8156,0.116010,0.000233,0.649873,0.184976,0.159393,0.642227,0.239530,0.830625
19,val_partial,7,1000,8656,0.114142,0.000225,0.624137,0.186393,0.158459,0.769502,0.350470,0.820781
20,val,7,1276,8932,0.114409,0.000220,0.495775,0.189195,0.145836,0.786682,0.354373,0.826179


## Loss

Train loss is the running mean over the epoch so far; validation is measured on
the held-out split. The gap between them widening while validation flattens is
the overfitting signal.

In [3]:
fig, ax = new_fig((6.5, 4))

handles = [
    plt.Line2D([], [], color=TRAIN_COLOR, marker="o", markersize=3.5, label="Train"),
    plt.Line2D([], [], color=VAL_COLOR, marker="o", markersize=3.5, label="Validation"),
]
# When full epochs exist, the mid-epoch checks ride underneath as a faint trace.
if len(partial) and not CURVE_IS_PARTIAL:
    ax.plot(partial["global_step"], partial["val_loss"], color=VAL_COLOR,
            linewidth=0.8, alpha=0.35)
    handles.append(plt.Line2D([], [], color=VAL_COLOR, alpha=0.35, linewidth=0.8,
                              label="Mid-epoch checks"))

ax.plot(curve["global_step"], curve["train_loss"], color=TRAIN_COLOR, marker="o",
        markersize=3.5, linewidth=1.6)
ax.plot(curve["global_step"], curve["val_loss"], color=VAL_COLOR, marker="o",
        markersize=3.5, linewidth=1.6)

ax.set_xlabel("Training Step")
ax.set_ylabel("BCE Loss")
ax.set_title("Training and Validation Loss"
             + (" (Mid-Epoch Checks)" if CURVE_IS_PARTIAL else ""))
left_legend(ax, handles, "", (1.02, 1.0))

save_fig(fig, "training_loss.png")
plt.show()

saved /home/vivian.chu/AETHER/figures/training_loss.png


## Validation AUROC and AUPRC

AUROC is the headline: threshold-free and unaffected by the 77/23 class split.
AUPRC gets its own figure rather than a shared axis — two measures on different
scales do not belong on one y-axis.

The dashed line is what a random ranker scores: 0.5 for AUROC, the positive rate
for AUPRC.

In [4]:
try:
    POS_RATE = pd.read_csv(REPO / "data/mhcattnnet/classI/val.csv")["bind"].mean()
except Exception:
    POS_RATE = None


def plot_val_metric(metric, title, chance, outname):
    col = f"val_{metric}"
    if col not in curve.columns or curve[col].notna().sum() == 0:
        print(f"{col} not logged yet — skipping")
        return

    series = curve.dropna(subset=[col])

    fig, ax = new_fig((6, 3.8))
    if len(partial) and not CURVE_IS_PARTIAL and col in partial.columns:
        ax.plot(partial["global_step"], partial[col], color=TAB["blue"],
                linewidth=0.8, alpha=0.35)
    ax.plot(series["global_step"], series[col], color=TAB["blue"], marker="o",
            markersize=3.5, linewidth=1.6)

    if chance is not None:
        ax.axhline(chance, color=REF_LINE, linestyle="--", linewidth=0.8)
        ax.text(0.99, chance, f"chance {chance:.3f} ", ha="right", va="bottom",
                transform=ax.get_yaxis_transform(), fontsize=8.5, color=REF_LINE)

    best = series[col].max()
    best_step = series.loc[series[col].idxmax(), "global_step"]
    ax.plot([best_step], [best], marker="o", markersize=7, color=TAB["red"], zorder=5)
    ax.annotate(f"best {best:.4f}", xy=(best_step, best), xytext=(0, 10),
                textcoords="offset points", ha="center", fontsize=9.5, color=TAB["red"])

    ax.set_xlabel("Training Step")
    ax.set_ylabel(metric.upper())
    ax.set_ylim(min(series[col].min(), chance or 1) - 0.04, 1.02)
    ax.set_title(title + (" (Mid-Epoch Checks)" if CURVE_IS_PARTIAL else ""))
    save_fig(fig, outname)
    plt.show()


plot_val_metric("auc", "Validation AUROC", 0.5, "training_val_auroc.png")

saved /home/vivian.chu/AETHER/figures/training_val_auroc.png


In [5]:
plot_val_metric("auprc", "Validation AUPRC", POS_RATE, "training_val_auprc.png")

saved /home/vivian.chu/AETHER/figures/training_val_auprc.png


In [6]:
# Learning-rate schedule actually taken — worth a look when a run stalls.
if "lr" in hist.columns and hist["lr"].notna().any():
    lr_hist = hist.dropna(subset=["lr"]).sort_values("global_step")
    fig, ax = new_fig((6, 2.6))
    ax.plot(lr_hist["global_step"], lr_hist["lr"], color=TAB["green"], linewidth=1.6)
    ax.set_xlabel("Training Step")
    ax.set_ylabel("Learning Rate")
    ax.set_title(f"Learning-Rate Schedule ({cfg['lr_scheduler']}, "
                 f"{cfg['warmup_steps']}-step warmup)")
    save_fig(fig, "training_lr_schedule.png")
    plt.show()

saved /home/vivian.chu/AETHER/figures/training_lr_schedule.png


## Test-Set Evaluation

Everything below uses `best.pt` — the checkpoint with the highest **validation**
AUROC — scored once on the held-out test split.

In [7]:
preds_path = RUN_DIR / "test_predictions.csv"
HAVE_TEST = preds_path.exists()

if not HAVE_TEST:
    print("No test_predictions.csv yet — the run has not reached the test stage.")
else:
    preds = pd.read_csv(preds_path)
    y_true = (preds["y_true"] >= 0.5).astype(int).to_numpy()
    y_prob = preds["y_pred_prob"].to_numpy()
    test_metrics = json.loads((RUN_DIR / "test_metrics.json").read_text())
    print(json.dumps(test_metrics, indent=2))

No test_predictions.csv yet — the run has not reached the test stage.


In [8]:
if HAVE_TEST:
    from sklearn.metrics import roc_auc_score, roc_curve

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)

    fig, ax = new_fig((4.6, 4.4))
    ax.plot(fpr, tpr, color=TAB["blue"], linewidth=1.8)
    ax.fill_between(fpr, tpr, color=TAB["blue"], alpha=0.12)
    ax.plot([0, 1], [0, 1], color=REF_LINE, linestyle="--", linewidth=0.8)
    ax.text(0.97, 0.05, f"AUROC {auc:.4f}", ha="right", fontsize=12,
            color=BAR_COLOR, transform=ax.transAxes)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.01)
    ax.set_title("ROC Curve, Test Split")
    save_fig(fig, "test_roc_curve.png")
    plt.show()

In [9]:
if HAVE_TEST:
    from sklearn.metrics import average_precision_score, precision_recall_curve

    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    base = y_true.mean()

    fig, ax = new_fig((4.6, 4.4))
    ax.plot(rec, prec, color=TAB["orange"], linewidth=1.8)
    ax.fill_between(rec, prec, color=TAB["orange"], alpha=0.12)
    ax.axhline(base, color=REF_LINE, linestyle="--", linewidth=0.8)
    ax.text(0.02, base + 0.015, f"chance {base:.3f}", fontsize=8.5, color=REF_LINE)
    ax.text(0.03, 0.05, f"AUPRC {ap:.4f}", fontsize=12, color=BAR_COLOR,
            transform=ax.transAxes)
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.01)
    ax.set_title("Precision-Recall Curve, Test Split")
    save_fig(fig, "test_pr_curve.png")
    plt.show()

In [10]:
if HAVE_TEST:
    fig, ax = new_fig()
    bins = np.linspace(0, 1, 51)
    ax.hist(y_prob[y_true == 1], bins=bins, color=BINDER, alpha=0.85, density=True)
    ax.hist(y_prob[y_true == 0], bins=bins, color=NON_BINDER, alpha=0.85, density=True)
    ax.set_xlabel("Predicted Probability")
    ax.set_ylabel("Density")
    ax.set_title("Score Separation by True Class")
    left_legend(ax, class_handles(), "", (1.02, 1.0))
    save_fig(fig, "test_score_separation.png")
    plt.show()

In [11]:
if HAVE_TEST:
    from sklearn.calibration import calibration_curve

    frac_pos, mean_pred = calibration_curve(y_true, y_prob, n_bins=12, strategy="quantile")

    fig, ax = new_fig((4.6, 4.4))
    ax.plot([0, 1], [0, 1], color=REF_LINE, linestyle="--", linewidth=0.8)
    ax.plot(mean_pred, frac_pos, color=TAB["teal"], marker="o", markersize=3.5,
            linewidth=1.6)
    ax.set_xlabel("Mean Predicted Probability")
    ax.set_ylabel("Observed Binder Fraction")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_title("Calibration (Above the Line = Under-Confident)")
    save_fig(fig, "test_calibration.png")
    plt.show()

## Per-Allele Performance

A pooled AUROC is dominated by the best-represented alleles. This breaks it out
for alleles with enough test rows to give a stable estimate, which is where a
model that looks fine overall can turn out to be carrying only the common ones.

In [12]:
if HAVE_TEST and "mhc_allele" in preds.columns:
    from sklearn.metrics import roc_auc_score

    MIN_ROWS = 200
    rows = []
    for allele, g in preds.groupby("mhc_allele"):
        yt = (g["y_true"] >= 0.5).astype(int)
        if len(g) >= MIN_ROWS and yt.nunique() == 2:
            rows.append({"allele": allele, "n": len(g),
                         "auc": roc_auc_score(yt, g["y_pred_prob"]),
                         "pos_rate": yt.mean()})
    print(f"{len(rows)} alleles with >= {MIN_ROWS} test rows and both classes")

if HAVE_TEST and "mhc_allele" in preds.columns and rows:
    per_allele = pd.DataFrame(rows).sort_values("auc")
    pooled = roc_auc_score(y_true, y_prob)

    fig, ax = new_fig((6.5, max(4, 0.23 * len(per_allele))))
    yy = np.arange(len(per_allele))
    ax.barh(yy, per_allele["auc"], height=0.68, color=BAR_COLOR)
    ax.axvline(0.5, color=REF_LINE, linestyle="--", linewidth=0.8)
    ax.axvline(pooled, color=TAB["red"], linewidth=1.2)
    ax.text(pooled, len(per_allele) + 0.4, f" pooled {pooled:.3f}", fontsize=9,
            color=TAB["red"], ha="left")
    for i, (a, n) in enumerate(zip(per_allele["auc"], per_allele["n"])):
        ax.text(a + 0.006, i, f"{a:.3f} (n={n:,})", va="center", fontsize=7.5)
    ax.set_yticks(yy, per_allele["allele"], fontsize=8)
    ax.set_xlabel("AUROC")
    ax.set_xlim(0.4, 1.08)
    ax.set_title("Per-Allele AUROC, Test Split")
    save_fig(fig, "test_per_allele_auc.png")
    plt.show()

    display(per_allele.sort_values("auc", ascending=False).reset_index(drop=True).round(4))
elif HAVE_TEST:
    # Small eval subsets leave no allele above the row threshold; that is not an
    # error, there is just nothing stable enough to break down.
    print("No allele has enough test rows for a per-allele estimate.")

## Run Summary

In [13]:
summary = {
    "Backbone":           cfg["model_name"],
    "Backbone Frozen":    cfg["freeze_backbone"],
    "MHC Precomputed":    cfg.get("precompute_mhc"),
    "Interaction Layers": cfg["num_interaction_layers"],
    "Self-Attention":     cfg["use_self_attn"],
    "Product Head":       cfg["use_product_head"],
    "Batch Size":         cfg["batch_size"],
    "Learning Rate":      cfg["lr"],
    "Epochs Run":         int(val["epoch"].max()) if len(val) else 0,
    "Best Val AUROC":     (round(float(curve["val_auc"].max()), 4)
                           if "val_auc" in curve and curve["val_auc"].notna().any() else None),
}
if HAVE_TEST:
    summary |= {f"Test {k.upper()}": round(v, 4) for k, v in test_metrics.items()}

pd.DataFrame({"Value": summary})

,Value
Backbone,facebook/esm2_t33_650M_UR50D
Backbone Frozen,True
MHC Precomputed,True
Interaction Layers,2
Self-Attention,True
Product Head,True
Batch Size,64
Learning Rate,0.0003
Epochs Run,7
Best Val AUROC,0.8201
